# CloudGuardian — Prioritization Model

CAP-CSE-3W Week 2 deliverable: a simple, transparent prioritization model over the consolidated Prowler
findings, built as `priority_score = severity_score x exposure_score x blast_radius_score`.

**Honesty note (per rubric — "justified features, honest evaluation"):** this is a heuristic scoring
model, not a trained ML classifier. Prowler's CSV schema does not include real CVSS scores or asset
criticality tags, so every feature below is a documented proxy built from fields Prowler *does* provide
(`SEVERITY`, `CHECK_ID`, `SERVICE_NAME`). The scoring weights and thresholds are hand-chosen and stated
explicitly rather than "learned," and every choice is justified in the markdown before it's used. This is
intentional: for a CSPM backlog of ~126 findings, a transparent, auditable heuristic that a security
engineer can second-guess is more useful — and more honest about its own limitations — than an opaque
model dressed up as ML.

In [1]:
import json
import pandas as pd

with open('consolidated_findings.json') as f:
    findings = json.load(f)

df = pd.DataFrame(findings)
fails = df[df['status'] == 'FAIL'].copy()
print(f"Total findings in consolidated DB: {len(df)}")
print(f"FAIL findings to prioritize: {len(fails)}")


Total findings in consolidated DB: 257
FAIL findings to prioritize: 126


## Feature 1 — `severity_score` (CVSS proxy)

Prowler assigns each check a qualitative severity (`CRITICAL` / `HIGH` / `MEDIUM` / `LOW`) but not a
numeric CVSS score. We map these onto the CVSS v3 severity bands' representative midpoints
(Critical 9.0–10.0, High 7.0–8.9, Medium 4.0–6.9, Low 0.1–3.9), picking a single representative value per
band rather than inventing false precision:

| Prowler severity | severity_score |
|---|---|
| CRITICAL | 10.0 |
| HIGH | 7.5 |
| MEDIUM | 5.0 |
| LOW | 2.5 |

This is a proxy, not a real CVSS lookup — disclosed here rather than presented as more rigorous than it
is.

In [2]:
SEVERITY_MAP = {'CRITICAL': 10.0, 'HIGH': 7.5, 'MEDIUM': 5.0, 'LOW': 2.5}
fails['severity_score'] = fails['severity'].map(SEVERITY_MAP).fillna(2.5)
print(fails['severity_score'].value_counts().sort_index(ascending=False))


severity_score
10.0     6
7.5     28
5.0     63
2.5     29
Name: count, dtype: int64


## Feature 2 — `exposure_score` (0.0-1.0)

Approximates "how directly can an external attacker reach this, without needing another foothold first."
Built as a keyword heuristic over `check_id` and `title`, since Prowler's schema has no explicit
network-reachability field we can query directly:

- **1.0** — check text mentions public/external/cross-account/internet reachability, or explicitly
  measures "no public access" / "no public IP" (i.e. the check *is* an exposure check).
- **0.8** — check concerns IAM privilege escalation (AdministratorAccess, wildcard actions, assume-role)
  — not network-reachable directly, but a stepping stone to full account compromise if the role is ever
  reached by any path.
- **0.3** — detective-control gaps (CloudTrail, Config, CloudWatch) — these don't expose anything
  themselves, but they reduce the chance of *catching* an attacker who exploited something else, so they
  are not scored at 0.
- **0.4** — everything else (default), e.g. encryption-at-rest gaps, which require a separate access
  vector (a leaked snapshot, a compromised credential) before they matter.

This is the least rigorous feature in the model and the most worth an engineer's second-guessing — flagged
explicitly rather than hidden behind a clean-looking number.

In [3]:
EXTERNAL_EXPOSURE_KEYWORDS = ['public', 'external', 'cross_account', 'internet', 'no_public_access', 'no_public_ip']
PRIVILEGE_EXPOSURE_KEYWORDS = ['administratoraccess', 'admin', 'assume_role', 'wildcard', 'privilege']

def exposure_score(row):
    text = (row['check_id'] + ' ' + row['title']).lower()
    if any(k in text for k in EXTERNAL_EXPOSURE_KEYWORDS):
        return 1.0
    if any(k in text for k in PRIVILEGE_EXPOSURE_KEYWORDS):
        return 0.8
    if row['service'] in ('cloudtrail', 'config', 'cloudwatch'):
        return 0.3
    return 0.4

fails['exposure_score'] = fails.apply(exposure_score, axis=1)
print(fails['exposure_score'].value_counts().sort_index(ascending=False))


exposure_score
1.0    18
0.8     5
0.4    82
0.3    21
Name: count, dtype: int64


## Feature 3 — `blast_radius_score` (1-3, integer)

Approximates "how much of the account is affected if this specific finding is exploited":

- **3 — account-wide.** IAM findings (a compromised/over-privileged role can touch anything its policies
  allow) and CloudTrail findings (missing logging degrades detection for the *entire* account, not one
  resource).
- **2 — multi-resource.** VPC/subnet and security-group findings — every instance launched into an
  affected subnet or protected by an affected SG inherits the exposure, not just one resource.
- **1 — single-resource.** Everything else (e.g. one S3 bucket's encryption, one RDS instance's
  encryption) — the blast radius is that one resource's data.

In [4]:
def blast_radius(row):
    if row['service'] in ('iam', 'cloudtrail'):
        return 3
    if row['service'] == 'vpc' or 'subnet' in row['check_id'].lower() or 'security_group' in row['check_id'].lower():
        return 2
    return 1

fails['blast_radius_score'] = fails.apply(blast_radius, axis=1)
print(fails['blast_radius_score'].value_counts().sort_index(ascending=False))


blast_radius_score
3    26
2     8
1    92
Name: count, dtype: int64


## Composite score

`priority_raw = severity_score x exposure_score x blast_radius_score`, then min-max scaled to 0-100 against
this scan's own maximum so the top finding in any given scan is always 100 (relative ranking within one
account/scan, not an absolute cross-account scale).

In [5]:
fails['priority_raw'] = fails['severity_score'] * fails['exposure_score'] * fails['blast_radius_score']
fails['priority_score'] = (fails['priority_raw'] / fails['priority_raw'].max() * 100).round(1)

ranked = fails.sort_values('priority_score', ascending=False)
cols = ['finding_id','check_id','resource_name','severity','exposure_score','blast_radius_score','priority_score','misconfig_id']
print(ranked[cols].head(15).to_string(index=False))


  finding_id                                                             check_id            resource_name severity  exposure_score  blast_radius_score  priority_score  misconfig_id
35120473ae3c                 iam_aws_attached_policy_no_administrative_privileges      AdministratorAccess CRITICAL             0.8                   3           100.0           NaN
33de767eddd8                                 iam_user_administrator_access_policy          terraform-admin CRITICAL             0.8                   3           100.0           NaN
015575453511                         iam_role_cross_account_readonlyaccess_policy    ref3tier-dev-web-role     HIGH             1.0                   3            93.8           7.0
c0d4a7e570ec                                  iam_role_administratoraccess_policy    ref3tier-dev-web-role     HIGH             0.8                   3            75.0           6.0
be7f507747fd                                   vpc_subnet_no_public_ip_by_default subnet-0

## Ranked view of just our 8 catalogued misconfigs

Confirms the model puts the two IAM findings (#6, #7) and the public-network findings (#1, #4) at the top
of the backlog, and the encryption-at-rest findings (#2, #5) and detective-control gaps (#8) lower — which
matches security intuition (an externally-reachable, account-wide privilege issue should outrank an
encryption-at-rest gap on a bucket nothing else depends on) and gives a sanity check that the heuristic
isn't producing nonsense rankings.

In [6]:
mc = ranked[ranked['misconfig_id'].notna()][cols]
print(mc.to_string(index=False))


  finding_id                                                  check_id            resource_name severity  exposure_score  blast_radius_score  priority_score  misconfig_id
015575453511              iam_role_cross_account_readonlyaccess_policy    ref3tier-dev-web-role     HIGH             1.0                   3            93.8           7.0
c0d4a7e570ec                       iam_role_administratoraccess_policy    ref3tier-dev-web-role     HIGH             0.8                   3            75.0           6.0
2eb9d32488b6                        vpc_subnet_no_public_ip_by_default subnet-084a2d306ddd4bd4f     HIGH             1.0                   2            62.5           4.0
97361d618901                                   s3_bucket_public_access  ref3tier-dev-app-kyll6s CRITICAL             1.0                   1            41.7           1.0
e7c2b08fb819                       s3_bucket_level_public_access_block  ref3tier-dev-app-kyll6s     HIGH             1.0                   1     

## Evaluation — honest limitations

- **No ground truth.** There is no labeled "this finding was actually exploited" dataset to validate
  against — this is a backlog-triage heuristic, not a model with measured precision/recall. Any claim of
  "accuracy" here would be fabricated.
- **Exposure heuristic is keyword-based**, not a real reachability graph (it doesn't actually trace
  security-group rules or route tables the way Prowler's own `rds_instance_no_public_access` check does —
  see the misconfig #3 finding in the catalogue, where a flag+SG combination that *looks* exposed by
  keyword is in fact network-mitigated). A more rigorous version would build an actual reachability graph
  from the VPC/SG/route-table data instead of pattern-matching check titles.
- **Severity mapping compresses information.** Two HIGH findings get the same `severity_score` even if
  one is "one step from full compromise" and the other is "best practice, low real risk" — Prowler's own
  qualitative severity already has this limitation, and this model inherits it rather than fixing it.
- **Weights are hand-chosen, not fit.** `0.8` for privilege-escalation exposure and `0.3` for
  detective-control gaps are defensible judgment calls, not calibrated against outcome data. Documented
  here so a reviewer can disagree with a specific number rather than the model as a black box.